# 🏢 Gestion des Clients - AskMe Search

Ce notebook permet de gérer les clients avec des index OpenSearch séparés.

**Use cases :**
- 🆕 Créer un nouveau client avec son index dédié
- 📋 Lister tous les clients existants
- 📊 Voir les statistiques par client
- 🗑️ Supprimer un client et ses données
- 🔄 Vider/réinitialiser un client

## 📦 Configuration

In [1]:
import sys
sys.path.append('../scripts')

from simple_indexer import SimpleIndexer, create_simple_index
import requests
import json
from datetime import datetime

# Configuration
OPENSEARCH_URL = "http://localhost:9200"

print("🏢 Notebook Gestion Clients chargé")
print(f"   OpenSearch: {OPENSEARCH_URL}")
print(f"   Convention: askme-[client-id]")

🏢 Notebook Gestion Clients chargé
   OpenSearch: http://localhost:9200
   Convention: askme-[client-id]


## 🔗 Test de Connexion

In [2]:
def test_opensearch_connection():
    """Vérifier la connexion à OpenSearch"""
    try:
        response = requests.get(f"{OPENSEARCH_URL}/_cluster/health")
        if response.status_code == 200:
            health = response.json()
            print(f"✅ OpenSearch connecté")
            print(f"   Status: {health.get('status', 'unknown')}")
            print(f"   Nœuds: {health.get('number_of_nodes', 0)}")
            return True
        return False
    except Exception as e:
        print(f"❌ Connexion impossible: {e}")
        return False

# Test de connexion
if test_opensearch_connection():
    print("\n🎉 Prêt à gérer les clients !")
else:
    print("\n⚠️ Vérifiez qu'OpenSearch est lancé: docker-compose up -d")

✅ OpenSearch connecté
   Status: green
   Nœuds: 1

🎉 Prêt à gérer les clients !


## 📋 Liste des Clients Existants

In [5]:
def list_all_clients():
    """Lister tous les clients (index askme-*)"""
    indexer = SimpleIndexer()
    all_indexes = indexer.list_indexes()
    
    clients = []
    for idx in all_indexes:
        if idx.startswith('askme-'):
            client_id = idx.replace('askme-', '')
            clients.append({'id': client_id, 'index': idx})
    
    return clients

def display_clients_stats():
    """Afficher les clients avec leurs statistiques"""
    clients = list_all_clients()
    
    if not clients:
        print("❌ Aucun client trouvé")
        return
    
    print(f"📋 {len(clients)} client(s) trouvé(s):")
    print("=" * 60)
    
    for client in clients:
        client_indexer = SimpleIndexer(index_name=client['index'])
        stats = client_indexer.get_index_stats()
        
        if stats:
            docs = stats.get('documents_count', 0)
            size = stats.get('size_mb', 0)
            print(f"🏢 {client['id']}")
            print(f"   📁 Index: {client['index']}")
            print(f"   📄 Documents: {docs}")
            print(f"   💾 Taille: {size} MB")
        else:
            print(f"🏢 {client['id']} (erreur stats)")
        print()

# Afficher les clients existants
display_clients_stats()

📋 1 client(s) trouvé(s):
🏢 avanteam-qualitysaas-dev
   📁 Index: askme-avanteam-qualitysaas-dev
   📄 Documents: 0
   💾 Taille: 0.0 MB



## 🆕 Créer un Nouveau Client

In [4]:
def create_new_client(client_id: str) -> bool:
    """Créer un nouveau client avec validation"""
    
    # Validation du nom
    if not client_id or not client_id.replace('-', '').replace('_', '').isalnum():
        print("❌ ID client invalide. Utilisez lettres, chiffres, - et _")
        return False
    
    # Vérifier si existe déjà
    existing_clients = list_all_clients()
    if any(c['id'] == client_id for c in existing_clients):
        print(f"⚠️ Client '{client_id}' existe déjà")
        return False
    
    # Créer l'index
    print(f"🏢 Création du client: {client_id}")
    success = SimpleIndexer.create_client_index(client_id, OPENSEARCH_URL)
    
    if success:
        print(f"✅ Client créé avec succès !")
        print(f"💡 Utilisation: SimpleIndexer(index_name='askme-{client_id}')")
        return True
    else:
        print(f"❌ Échec de la création")
        return False

# Exemples de création (décommentez pour utiliser)
print("🆕 Créer des clients de test:")
print("="*40)

# DÉCOMMENTEZ POUR CRÉER:
create_new_client("avanteam-qualitysaas-dev")
# create_new_client("cabinet-avocat") 
# create_new_client("startup-tech")

print("⚠️ Décommentez les lignes ci-dessus pour créer des clients")

🆕 Créer des clients de test:
🏢 Création du client: avanteam-qualitysaas-dev
🏢 Création de l'index client: askme-avanteam-qualitysaas-dev
✅ Index askme-avanteam-qualitysaas-dev créé
✅ Index client créé: askme-avanteam-qualitysaas-dev
💡 Utilisation: SimpleIndexer(index_name='askme-avanteam-qualitysaas-dev')
✅ Client créé avec succès !
💡 Utilisation: SimpleIndexer(index_name='askme-avanteam-qualitysaas-dev')
⚠️ Décommentez les lignes ci-dessus pour créer des clients


## 🔧 Opérations sur un Client

In [7]:
def get_client_details(client_id: str):
    """Obtenir les détails complets d'un client"""
    index_name = f"askme-{client_id}"
    client_indexer = SimpleIndexer(index_name=index_name)
    
    print(f"🏢 Détails du client: {client_id}")
    print("=" * 50)
    
    # Statistiques générales
    stats = client_indexer.get_index_stats()
    if stats:
        print(f"📊 Statistiques:")
        print(f"   📄 Documents: {stats.get('documents_count', 0)}")
        print(f"   💾 Taille: {stats.get('size_mb', 0)} MB")
        print(f"   💾 Taille brute: {stats.get('size_bytes', 0)} bytes")
    
    # Liste des documents
    docs_response = client_indexer.list_documents(size=50)
    if docs_response and 'hits' in docs_response:
        hits = docs_response['hits']['hits']
        
        # Grouper par fichier
        files = {}
        for hit in hits:
            source = hit['_source']
            filepath = source.get('filepath', 'Inconnu')
            title = source.get('title', 'Sans titre')
            access_rights = source.get('accessRights', None)
            
            if filepath not in files:
                files[filepath] = {
                    'title': title,
                    'chunks': 0,
                    'access_rights': access_rights
                }
            files[filepath]['chunks'] += 1
        
        print(f"\n📚 Documents ({len(files)} fichiers):")
        for filepath, info in files.items():
            rights_str = f" - 🔐 {info['access_rights']}" if info['access_rights'] else " - 🌐 Libre"
            print(f"   📄 {info['title']} ({info['chunks']} chunks){rights_str}")

# Tester avec un client existant
clients = list_all_clients()
if clients:
    get_client_details("avanteam-qualitysaas-dev")
else:
    print("❌ Aucun client à analyser. Créez-en un d'abord !")

🏢 Détails du client: avanteam-qualitysaas-dev
📊 Statistiques:
   📄 Documents: 0
   💾 Taille: 0.0 MB
   💾 Taille brute: 208 bytes

📚 Documents (0 fichiers):


## 🔄 Vider/Réinitialiser un Client

In [24]:
def reset_client(client_id: str) -> bool:
    """Vider complètement un client (supprimer + recréer)"""
    
    # Vérifier que le client existe
    existing_clients = list_all_clients()
    if not any(c['id'] == client_id for c in existing_clients):
        print(f"❌ Client '{client_id}' introuvable")
        return False
    
    print(f"🔄 Réinitialisation du client: {client_id}")
    print("⚠️ ATTENTION: Toutes les données seront perdues !")
    
    # Statistiques avant suppression
    index_name = f"askme-{client_id}"
    client_indexer = SimpleIndexer(index_name=index_name)
    stats_before = client_indexer.get_index_stats()
    
    if stats_before:
        docs_before = stats_before.get('documents_count', 0)
        size_before = stats_before.get('size_mb', 0)
        print(f"📊 Avant: {docs_before} documents, {size_before} MB")
    
    # Supprimer l'index
    step1 = SimpleIndexer.delete_client_index(client_id, OPENSEARCH_URL)
    
    if step1:
        # Recréer l'index vide
        step2 = SimpleIndexer.create_client_index(client_id, OPENSEARCH_URL)
        
        if step2:
            print(f"✅ Client réinitialisé avec succès !")
            print(f"🆕 Index vide prêt à être utilisé")
            return True
    
    print(f"❌ Échec de la réinitialisation")
    return False

# Exemple de réinitialisation (ATTENTION: destructif !)
print("🔄 Réinitialisation de client:")
print("="*40)
print("⚠️ DÉCOMMENTEZ UNIQUEMENT SI VOUS ÊTES SÛR !")

# DÉCOMMENTEZ POUR RÉINITIALISER (remplacez 'nom-client'):
reset_client("avanteam-qualitysaas-dev")

print("💡 Remplacez 'nom-client' par l'ID du client à réinitialiser")

🔄 Réinitialisation de client:
⚠️ DÉCOMMENTEZ UNIQUEMENT SI VOUS ÊTES SÛR !
🔄 Réinitialisation du client: avanteam-qualitysaas-dev
⚠️ ATTENTION: Toutes les données seront perdues !
📊 Avant: 0 documents, 0.0 MB
✅ Index client supprimé: askme-avanteam-qualitysaas-dev
🏢 Création de l'index client: askme-avanteam-qualitysaas-dev
✅ Index askme-avanteam-qualitysaas-dev créé
✅ Index client créé: askme-avanteam-qualitysaas-dev
💡 Utilisation: SimpleIndexer(index_name='askme-avanteam-qualitysaas-dev')
✅ Client réinitialisé avec succès !
🆕 Index vide prêt à être utilisé
💡 Remplacez 'nom-client' par l'ID du client à réinitialiser


## 🗑️ Supprimer un Client

In [7]:
def delete_client_permanently(client_id: str) -> bool:
    """Supprimer définitivement un client"""
    
    # Vérifier que le client existe
    existing_clients = list_all_clients()
    if not any(c['id'] == client_id for c in existing_clients):
        print(f"❌ Client '{client_id}' introuvable")
        return False
    
    print(f"🗑️ Suppression définitive du client: {client_id}")
    print("⚠️ ATTENTION: Cette action est IRRÉVERSIBLE !")
    
    # Afficher les données qui seront perdues
    index_name = f"askme-{client_id}"
    client_indexer = SimpleIndexer(index_name=index_name)
    stats = client_indexer.get_index_stats()
    
    if stats:
        docs = stats.get('documents_count', 0)
        size = stats.get('size_mb', 0)
        print(f"📊 Données à supprimer: {docs} documents, {size} MB")
    
    # Supprimer l'index
    success = SimpleIndexer.delete_client_index(client_id, OPENSEARCH_URL)
    
    if success:
        print(f"✅ Client supprimé définitivement")
        print(f"🗑️ Index 'askme-{client_id}' n'existe plus")
        return True
    else:
        print(f"❌ Échec de la suppression")
        return False

# Exemple de suppression (TRÈS DESTRUCTIF !)
print("🗑️ Suppression définitive:")
print("="*40)
print("⚠️ DÉCOMMENTEZ UNIQUEMENT EN CONNAISSANCE DE CAUSE !")

# DÉCOMMENTEZ POUR SUPPRIMER DÉFINITIVEMENT:
delete_client_permanently("notebook02-test")

print("💡 Remplacez 'nom-client' par l'ID du client à supprimer")
print("⚠️ Cette action est irréversible !")

🗑️ Suppression définitive:
⚠️ DÉCOMMENTEZ UNIQUEMENT EN CONNAISSANCE DE CAUSE !
🗑️ Suppression définitive du client: notebook02-test
⚠️ ATTENTION: Cette action est IRRÉVERSIBLE !
📊 Données à supprimer: 3 documents, 0.01 MB
✅ Index client supprimé: askme-notebook02-test
✅ Client supprimé définitivement
🗑️ Index 'askme-notebook02-test' n'existe plus
💡 Remplacez 'nom-client' par l'ID du client à supprimer
⚠️ Cette action est irréversible !


## 📋 Résumé - Gestion des Clients

✅ **Fonctions disponibles dans ce notebook :**

### 🔍 **Consultation**
- `list_all_clients()` - Lister tous les clients
- `display_clients_stats()` - Afficher avec statistiques
- `get_client_details(client_id)` - Détails complets d'un client

### 🆕 **Création**
- `create_new_client(client_id)` - Créer un nouveau client
- Validation automatique des noms
- Vérification des doublons

### 🔧 **Maintenance**
- `reset_client(client_id)` - Vider un client (garder l'index)
- `delete_client_permanently(client_id)` - Suppression définitive

### 💡 **Usage dans votre code**
```python
# Créer un client
SimpleIndexer.create_client_index("mon-client")

# Utiliser l'indexeur client
indexer = SimpleIndexer(index_name="askme-mon-client")

# Vider le client
SimpleIndexer.delete_client_index("mon-client")
SimpleIndexer.create_client_index("mon-client")
```

🚀 **Le système multi-clients est maintenant opérationnel !**